In [1]:
import os
import re
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from peft import PeftModel
from transformers import AutoTokenizer, AutoConfig, AutoModel

In [2]:
MODEL_DIR = "/content/B1_light_best_model"   # sửa lại path của bạn
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
with open(os.path.join(MODEL_DIR, "export_meta.json"), "r", encoding="utf-8") as f:
    meta = json.load(f)

BASE_MODEL_NAME = meta["model_name"]
MAX_LENGTH = meta["max_length"]
TARGET_COLS = meta["target_cols"]
GRA_FEATURE_COLS = meta["gra_feature_cols"]

gra_feat_mean = np.array(meta["gra_feat_mean"], dtype=np.float32)
gra_feat_std = np.array(meta["gra_feat_std"], dtype=np.float32)
gra_feat_std = np.where(gra_feat_std < 1e-6, 1.0, gra_feat_std)

print("BASE_MODEL_NAME =", BASE_MODEL_NAME)
print("MAX_LENGTH =", MAX_LENGTH)
print("TARGET_COLS =", TARGET_COLS)
print("Num grammar features =", len(GRA_FEATURE_COLS))

BASE_MODEL_NAME = Qwen/Qwen2.5-3B-Instruct
MAX_LENGTH = 1024
TARGET_COLS = ['TR', 'CC', 'LR', 'GRA']
Num grammar features = 11


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [5]:
def build_input_text(prompt: str, essay: str) -> str:
    prompt = str(prompt).strip()
    essay = str(essay).strip()
    return f"[PROMPT]\n{prompt}\n\n[ESSAY]\n{essay}"


def extract_grammar_features(text: str):
    text = str(text).strip()

    words = re.findall(r"\b[\w']+\b", text)
    word_count = max(len(words), 1)

    sentence_candidates = re.split(r"[.!?]+", text)
    sentences = [s.strip() for s in sentence_candidates if s.strip()]
    sentence_count = max(len(sentences), 1)

    avg_sentence_len = word_count / sentence_count

    sent_word_counts = []
    for s in sentences:
        sw = re.findall(r"\b[\w']+\b", s)
        sent_word_counts.append(len(sw))

    if len(sent_word_counts) == 0:
        sent_word_counts = [word_count]

    short_sentence_ratio = np.mean([c < 8 for c in sent_word_counts])
    long_sentence_ratio = np.mean([c > 25 for c in sent_word_counts])

    punct_count = len(re.findall(r"[,:;()\-\"]", text))
    punct_density = punct_count / word_count

    repeated_punct_count = len(re.findall(r"([!?.,;:])\1+", text))
    repeated_punct_ratio = repeated_punct_count / sentence_count

    lowercase_sent_starts = 0
    for s in sentences:
        s = s.strip()
        if len(s) > 0 and s[0].islower():
            lowercase_sent_starts += 1
    lowercase_sent_start_ratio = lowercase_sent_starts / sentence_count

    lowercase_i_count = len(re.findall(r"\bi\b", text))
    lowercase_i_ratio = lowercase_i_count / word_count

    repeated_word_count = 0
    lower_words = [w.lower() for w in words]
    for i in range(1, len(lower_words)):
        if lower_words[i] == lower_words[i - 1]:
            repeated_word_count += 1
    repeated_word_ratio = repeated_word_count / word_count

    missing_terminal_punct = 0.0 if re.search(r"[.!?]\s*$", text) else 1.0

    feats = np.array([
        float(word_count),
        float(sentence_count),
        float(avg_sentence_len),
        float(short_sentence_ratio),
        float(long_sentence_ratio),
        float(punct_density),
        float(repeated_punct_ratio),
        float(lowercase_sent_start_ratio),
        float(lowercase_i_ratio),
        float(repeated_word_ratio),
        float(missing_terminal_punct),
    ], dtype=np.float32)

    return feats


def normalize_gra_features(feats: np.ndarray) -> np.ndarray:
    return (feats - gra_feat_mean) / gra_feat_std

In [6]:
class QwenForIELTSMultiTaskInference(nn.Module):
    def __init__(self, base_model_name: str, model_dir: str, tokenizer, head_dropout: float = 0.1):
        super().__init__()

        self.config = AutoConfig.from_pretrained(base_model_name)
        self.config.pad_token_id = tokenizer.pad_token_id

        # Dùng float32 cho inference để tránh lỗi mismatch dtype
        base_model = AutoModel.from_pretrained(
            base_model_name,
            torch_dtype=torch.float32,
        )
        base_model.config.pad_token_id = tokenizer.pad_token_id
        base_model.config.use_cache = True

        # load LoRA adapter
        self.backbone = PeftModel.from_pretrained(base_model, model_dir)

        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(head_dropout)

        self.head_tr = nn.Linear(hidden_size, 1)
        self.head_cc = nn.Linear(hidden_size, 1)
        self.head_lr = nn.Linear(hidden_size, 1)

        self.gra_feat_dim = len(GRA_FEATURE_COLS)
        self.gra_mlp = nn.Sequential(
            nn.Linear(hidden_size + self.gra_feat_dim, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )

        self.reg_activation = nn.Sigmoid()

        # load custom heads
        head_state = torch.load(
            os.path.join(model_dir, "custom_heads.pt"),
            map_location="cpu"
        )
        self.head_tr.load_state_dict(head_state["head_tr"])
        self.head_cc.load_state_dict(head_state["head_cc"])
        self.head_lr.load_state_dict(head_state["head_lr"])
        self.gra_mlp.load_state_dict(head_state["gra_mlp"])

        # ép toàn bộ heads sang float32 luôn
        self.head_tr = self.head_tr.float()
        self.head_cc = self.head_cc.float()
        self.head_lr = self.head_lr.float()
        self.gra_mlp = self.gra_mlp.float()

    def _last_token_pool(self, hidden_states, attention_mask):
        last_token_idx = attention_mask.sum(dim=1) - 1
        last_token_idx = last_token_idx.clamp(min=0)
        batch_idx = torch.arange(hidden_states.size(0), device=hidden_states.device)
        pooled = hidden_states[batch_idx, last_token_idx]
        return pooled

    def forward(self, input_ids=None, attention_mask=None, gra_features=None):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state
        pooled = self._last_token_pool(hidden_states, attention_mask)
        pooled = self.dropout(pooled)

        # đảm bảo pooled là float32
        pooled = pooled.float()

        if gra_features is None:
            raise ValueError("gra_features is required")

        gra_features = gra_features.to(device=pooled.device, dtype=pooled.dtype)
        gra_input = torch.cat([pooled, gra_features], dim=1)

        tr = self.reg_activation(self.head_tr(pooled))
        cc = self.reg_activation(self.head_cc(pooled))
        lr = self.reg_activation(self.head_lr(pooled))
        gra = self.reg_activation(self.gra_mlp(gra_input))

        logits = torch.cat([tr, cc, lr, gra], dim=1)
        return logits

In [7]:
model = QwenForIELTSMultiTaskInference(
    base_model_name=BASE_MODEL_NAME,
    model_dir=MODEL_DIR,
    tokenizer=tokenizer,
    head_dropout=meta.get("head_dropout", 0.1),
)

model.to(DEVICE)
model.eval()

print("Model loaded on:", DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded on: cuda


In [8]:
def round_to_half(x):
    return round(x * 2) / 2.0


@torch.no_grad()
def predict_one(prompt: str, essay: str, round_band: bool = True):
    text = build_input_text(prompt, essay)

    gra_feats = extract_grammar_features(essay)
    gra_feats = normalize_gra_features(gra_feats)

    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    gra_features = torch.tensor(gra_feats, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        gra_features=gra_features
    )  # shape [1, 4], range 0..1

    preds_scaled = logits.squeeze(0).detach().float().cpu().numpy()
    preds_band = preds_scaled * 9.0

    result = {}
    for i, col in enumerate(TARGET_COLS):
        raw_score = float(preds_band[i])
        result[f"{col}_raw"] = raw_score
        result[col] = round_to_half(raw_score) if round_band else raw_score

    result["overall_raw_mean"] = float(np.mean(preds_band))
    result["overall"] = round_to_half(result["overall_raw_mean"]) if round_band else result["overall_raw_mean"]

    return result

In [15]:
prompt = "Some people think that zoos are cruel and should be closed down. Others, however, believe that zoos can be useful in protecting wild animals. Discuss both views and give your opinion."

essay = """
Some people argue that zoos help to preserve wild creatures , while others say that they are inhumane and should be abolished . While the development of breeding programmes contributes to the preservation of endangered species , I believe that the poor conditions that many animals held in captivity are kept in make the existence of zoos unacceptable .

On the one hand, there are many projects in existence in zoological parks around the world where species facing extinction have been successfully bred in captivity and their numbers increased substantially . This is important for ensuring the survival of animals under threat from poaching and the destruction of their natural environments . A good example of this is the golden lion tamarin from Brazil which nearly died out because of logging and mining activities which are destroying its habitat. Today, a third of wild golden lion tamarins were raised in captivity.

On the other hand, a significant percentage of zoos house their animals in cramped cages with very little space to move around or behave naturally . This can lead to them becoming distressed and depressed as well as suffering physically through lack of exercise. A friend of mine recently visited a wildlife park while on holiday abroad and was very upset to see the lions pacing up and down in a narrow , bare pen and eagles in enclosures so small that they were unable to fly.

In conclusion, although zoos do help to safeguard dwindling populations of particular species, the suffering experienced by many captive creatures due to unsuitable living conditions amounts to cruelty and they should not be allowed to exist.
"""

pred = predict_one(prompt, essay, round_band=True)
pred

{'TR_raw': 6.366105079650879,
 'TR': 6.5,
 'CC_raw': 6.410579204559326,
 'CC': 6.5,
 'LR_raw': 6.3201751708984375,
 'LR': 6.5,
 'GRA_raw': 6.198092460632324,
 'GRA': 6.0,
 'overall_raw_mean': 6.323738098144531,
 'overall': 6.5}